<a href="https://colab.research.google.com/github/vmyel/thesis_ref/blob/main/pd_main_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas

In [ ]:
pip install openpyxl

In [ ]:
pip install scikit-learn

In [ ]:
pip install numpy

In [ ]:
pip install matplotlib

In [ ]:
pip install seaborn

In [ ]:
# ============================================================
# 0.  Mount Google Drive & install/import libraries
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1.  USER-DEFINED PATHS  –– adjust to your Drive layout
# ============================================================
METADATA_PATH = 'PaHaW/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
SVC_ROOT = 'PaHaW/PaHaW/PaHaW_public/'

# METADATA_PATH = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
# SVC_ROOT = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_public/'
# ============================================================
# 2.  Load & inspect metadata
# ============================================================
meta = pd.read_excel(METADATA_PATH, dtype={'ID': str})

# Normalise column names (strip whitespace, consistent case)
meta.columns = meta.columns.str.strip()

# Make sure the subject ID column is zero-padded to 5 chars
# (adjust 'ID' to whatever the actual column header is)
meta['ID'] = meta['ID'].astype(str).str.zfill(5)

print("Metadata shape :", meta.shape)
print("Columns        :", meta.columns.tolist())

# ============================================================
# 3.  Filter out SEVERE stages  (H&Y / UPDRS V  >= 4.0)
#     NaN means Healthy Control → must be preserved
# ============================================================
before = len(meta)

# Keep row if UPDRS V is NaN (HC) OR score is below 4.0
meta_filtered = meta[meta['UPDRS V'].isna() | (meta['UPDRS V'] < 4.0)].copy()

after = len(meta_filtered)

print(f"Removed {before - after} subject(s) with UPDRS V >= 4.0")
print(f"Remaining subjects: {after}")   # should now be ~72

# Quick sanity check
print("\nUPDRS V distribution after filter:")
print(meta_filtered['UPDRS V'].value_counts(dropna=False).sort_index())

# ============================================================
# 4.  Assign group labels
#       • Healthy Controls : UPDRS V is NaN  (no H&Y score)
#       • Early PD         : UPDRS V  1.0 – 2.0
#       • Moderate PD      : UPDRS V  2.5 – 3.0
# ============================================================
def assign_group(row):
    score = row['UPDRS V']

    # Healthy controls have no UPDRS V score → NaN
    if pd.isna(score):
        return 'Healthy Control'
    elif 1.0 <= score <= 2.0:
        return 'Early PD'
    elif 2.5 <= score <= 3.0:
        return 'Moderate PD'
    else:
        return 'Unknown'   # safety catch

meta_filtered['Group'] = meta_filtered.apply(assign_group, axis=1)

print("Group distribution:")
print(meta_filtered['Group'].value_counts())
print()
print(meta_filtered[['ID', 'Disease', 'UPDRS V', 'Group']].head(72).to_string(index=False))

# ============================================================
# 5.  Parse a single SVC file
# ============================================================
def parse_svc(filepath: str) -> pd.DataFrame:
    """
    SVC format
    ----------
    Line 1   : number of samples (integer)
    Lines 2+ : Y  X  timestamp  button_state  azimuth  altitude  pressure
    """
    with open(filepath, 'r') as fh:
        lines = fh.read().splitlines()

    n_samples = int(lines[0].strip())
    data_lines = lines[1: n_samples + 1]          # safety slice

    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        records.append({
            'y'            : float(parts[0]),
            'x'            : float(parts[1]),
            'timestamp'    : float(parts[2]),
            'button_state' : int(parts[3]),
            'azimuth'      : float(parts[4]),
            'altitude'     : float(parts[5]),
            'pressure'     : float(parts[6]),
        })

    df = pd.DataFrame(records)
    df['n_declared'] = n_samples
    return df


# ============================================================
# 6.  Load ALL SVC files  (task number only, no session)
#     Filename format:  {subjectID}_{task}_{repetition}
#     e.g.  00098_6_1  →  subject=00098, task=6
# ============================================================
all_records = []
valid_ids   = set(meta_filtered['ID'].tolist())

for subj_folder in sorted(Path(SVC_ROOT).iterdir()):
    if not subj_folder.is_dir():
        continue

    subj_id = subj_folder.name.zfill(5)   # e.g. '98' → '00098'

    if subj_id not in valid_ids:
        continue

    meta_row = meta_filtered[meta_filtered['ID'] == subj_id].iloc[0]

    svc_files = sorted([f for f in subj_folder.iterdir() if f.is_file()])

    for svc_path in svc_files:
        try:
            svc_df = parse_svc(str(svc_path))
        except Exception as e:
            print(f"  [WARN] Could not parse {svc_path.name}: {e}")
            continue

        # e.g. '00098_6_1' → parts = ['00098', '6', '1']
        parts = svc_path.stem.split('_')
        task  = parts[1] if len(parts) > 1 else 'unknown'
        # parts[2] is just the repetition suffix ('1'), not used

        svc_df['subject_id'] = subj_id
        svc_df['task']       = task          # task number (1-8)
        svc_df['file_name']  = svc_path.name

        # Attach metadata
        svc_df['group']      = meta_row['Group']
        svc_df['updrs_v']    = meta_row['UPDRS V']
        svc_df['disease']    = meta_row['Disease']
        svc_df['age']        = meta_row['Age']
        svc_df['sex']        = meta_row['Sex']

        all_records.append(svc_df)

full_df = pd.concat(all_records, ignore_index=True)

print(f"Total stroke-point rows : {len(full_df):,}")
print(f"Unique subjects         : {full_df['subject_id'].nunique()}")
print(f"Unique SVC files        : {full_df['file_name'].nunique()}")
print(f"\nTasks found             : {sorted(full_df['task'].unique())}")

In [ ]:
# ============================================================
# 4.2  Patient-Level Stratified 5-Fold Cross-Validation Setup
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

# --- Define two classification objectives ---

# Objective 1: Detection (Healthy Control vs PD)
def assign_detection_label(group):
    if group == 'Healthy Control':
        return 0
    else:
        return 1  # Early PD or Moderate PD

# Objective 2: Staging (Early PD vs Moderate PD) — only PD subjects
def assign_staging_label(group):
    if group == 'Early PD':
        return 0
    elif group == 'Moderate PD':
        return 1
    else:
        return None  # Healthy controls excluded

# --- Build a subject-level dataframe for fold assignment ---
subject_df = meta_filtered[['ID', 'Group']].copy()
subject_df['detection_label'] = subject_df['Group'].apply(assign_detection_label)
subject_df['staging_label'] = subject_df['Group'].apply(assign_staging_label)

print("=" * 60)
print("DETECTION TASK — Label Distribution (subject-level)")
print("=" * 60)
print(subject_df['detection_label'].value_counts().rename({0: 'Healthy (0)', 1: 'PD (1)'}))
print()

staging_subject_df = subject_df.dropna(subset=['staging_label']).copy()
staging_subject_df['staging_label'] = staging_subject_df['staging_label'].astype(int)
print("=" * 60)
print("STAGING TASK — Label Distribution (subject-level)")
print("=" * 60)
print(staging_subject_df['staging_label'].value_counts().rename({0: 'Early PD (0)', 1: 'Moderate PD (1)'}))
print()

# --- Create the 5-Fold splitter ---
N_SPLITS = 5
RANDOM_STATE = 42

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- Build file-level dataframes for each task ---
# Each SVC file is one sample; the group key is subject_id

file_level_df = full_df.groupby('file_name').first().reset_index()[
    ['file_name', 'subject_id', 'task', 'group', 'updrs_v', 'disease']
]
file_level_df['detection_label'] = file_level_df['group'].apply(assign_detection_label)
file_level_df['staging_label'] = file_level_df['group'].apply(assign_staging_label)

# Detection file-level
detect_file_df = file_level_df.copy()

# Staging file-level (PD only)
stage_file_df = file_level_df.dropna(subset=['staging_label']).copy()
stage_file_df['staging_label'] = stage_file_df['staging_label'].astype(int)

print(f"Detection samples (files): {len(detect_file_df)}")
print(f"Staging samples (files)  : {len(stage_file_df)}")
print()

# --- Generate and display fold splits ---
def generate_folds(df, label_col, task_name):
    """Generate fold indices and print summary."""
    X = df.index.values
    y = df[label_col].values
    groups = df['subject_id'].values

    folds = []
    print(f"{'='*60}")
    print(f"Fold Summary for: {task_name}")
    print(f"{'='*60}")
    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
        train_subjects = set(df.iloc[train_idx]['subject_id'].unique())
        test_subjects = set(df.iloc[test_idx]['subject_id'].unique())
        overlap = train_subjects & test_subjects

        print(f"\nFold {fold_idx + 1}:")
        print(f"  Train: {len(train_idx)} files, {len(train_subjects)} subjects")
        print(f"  Test : {len(test_idx)} files, {len(test_subjects)} subjects")
        print(f"  Train label dist: {dict(pd.Series(y[train_idx]).value_counts().sort_index())}")
        print(f"  Test  label dist: {dict(pd.Series(y[test_idx]).value_counts().sort_index())}")
        print(f"  Patient leakage : {'NONE ✓' if len(overlap) == 0 else f'WARNING: {overlap}'}")

        folds.append((train_idx, test_idx))
    return folds

detection_folds = generate_folds(detect_file_df, 'detection_label', 'DETECTION')
staging_folds = generate_folds(stage_file_df, 'staging_label', 'STAGING')

In [ ]:
# ============================================================
# 4.3  Data Preprocessing
# ============================================================
from sklearn.preprocessing import StandardScaler
from scipy.stats import median_abs_deviation

# ============================================================
# 4.3 PATH A — Deep Learning Preprocessing
# ============================================================

SEQUENCE_LENGTH = 500  # L = 500 time-steps
DL_RAW_FEATURES = ['x', 'y', 'pressure', 'button_state']

def compute_dl_derived_features(file_data):
    """
    Compute 16 per-time-step derived features from raw handwriting data.
    These are the temporal equivalents of the 16 ML statistical features.
    
    Returns a DataFrame with original 4 + 16 derived = 20 features per time-step.
    """
    df = file_data[DL_RAW_FEATURES + ['timestamp']].copy()
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    dt = df['timestamp'].diff().fillna(1e-6)
    dt = dt.replace(0, 1e-6)
    
    dx = df['x'].diff().fillna(0)
    dy = df['y'].diff().fillna(0)
    
    # ---- Velocity (4 features) ----
    df['vx']       = dx / dt
    df['vy']       = dy / dt
    df['velocity'] = np.sqrt(df['vx']**2 + df['vy']**2)
    df['displacement'] = np.sqrt(dx**2 + dy**2)
    
    # ---- Acceleration (4 features) ----
    dvx = df['vx'].diff().fillna(0)
    dvy = df['vy'].diff().fillna(0)
    df['ax']           = dvx / dt
    df['ay']           = dvy / dt
    df['acceleration'] = np.sqrt(df['ax']**2 + df['ay']**2)
    df['velocity_change'] = df['velocity'].diff().fillna(0)
    
    # ---- Jerk (4 features) ----
    dax = df['ax'].diff().fillna(0)
    day = df['ay'].diff().fillna(0)
    df['jx']   = dax / dt
    df['jy']   = day / dt
    df['jerk'] = np.sqrt(df['jx']**2 + df['jy']**2)
    df['acceleration_change'] = df['acceleration'].diff().fillna(0)
    
    # ---- Pressure dynamics (4 features) ----
    df['pressure_change']     = df['pressure'].diff().fillna(0) / dt
    df['pressure_abs_change'] = df['pressure'].diff().fillna(0).abs()
    df['angle']               = np.arctan2(dy, dx)
    df['angular_velocity']    = df['angle'].diff().fillna(0) / dt
    
    # Replace inf/nan with 0
    df = df.replace([np.inf, -np.inf], 0).fillna(0)
    
    return df


# Define the full 20-feature list
DL_FEATURES = [
    'x', 'y', 'pressure', 'button_state',
    'vx', 'vy', 'velocity', 'displacement',
    'ax', 'ay', 'acceleration', 'velocity_change',
    'jx', 'jy', 'jerk', 'acceleration_change',
    'pressure_change', 'pressure_abs_change', 'angle', 'angular_velocity'
]

print(f"DL features per time-step: {len(DL_FEATURES)}")


def extract_dl_sequence(file_data):
    """
    Extract a time-series matrix with all 20 features for a single SVC file.
    Returns shape (n_timesteps, 20).
    """
    df = compute_dl_derived_features(file_data)
    seq = df[DL_FEATURES].values.astype(np.float32)
    return seq


def pad_or_clip_sequence(seq, target_length=None):
    """
    Clip if longer than target_length, zero-pad if shorter.
    Returns shape (target_length, n_features).
    """
    if target_length is None:
        target_length = SEQUENCE_LENGTH
        
    n_timesteps, n_features = seq.shape
    if n_timesteps >= target_length:
        return seq[:target_length]
    else:
        padded = np.zeros((target_length, n_features), dtype=np.float32)
        padded[:n_timesteps] = seq
        return padded


def prepare_dl_data(full_df, file_df, train_idx, test_idx, label_col):
    """
    Full Path A pipeline for a single fold.
    Now uses 20 features per time-step (4 raw + 16 derived).
    """
    train_files = file_df.iloc[train_idx]['file_name'].values
    test_files = file_df.iloc[test_idx]['file_name'].values

    print("  [DL] Extracting sequences with derived features...")
    train_sequences = []
    for fname in train_files:
        fdata = full_df[full_df['file_name'] == fname]
        seq = extract_dl_sequence(fdata)
        train_sequences.append(seq)

    test_sequences = []
    for fname in test_files:
        fdata = full_df[full_df['file_name'] == fname]
        seq = extract_dl_sequence(fdata)
        test_sequences.append(seq)

    # Fit Z-score scaler on TRAINING data only
    all_train_points = np.vstack(train_sequences)
    scaler = StandardScaler()
    scaler.fit(all_train_points)

    train_scaled = [scaler.transform(seq) for seq in train_sequences]
    test_scaled = [scaler.transform(seq) for seq in test_sequences]

    X_train = np.array([pad_or_clip_sequence(seq, SEQUENCE_LENGTH) for seq in train_scaled])
    X_test = np.array([pad_or_clip_sequence(seq, SEQUENCE_LENGTH) for seq in test_scaled])

    y_train = file_df.iloc[train_idx][label_col].values.astype(np.int64)
    y_test = file_df.iloc[test_idx][label_col].values.astype(np.int64)

    print(f"  [DL] Features per time-step: {X_train.shape[2]} (4 raw + 16 derived)")
    print(f"  [DL] Sequence Length L = {SEQUENCE_LENGTH}")
    print(f"  [DL] X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"  [DL] y_train dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  [DL] y_test  dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

    return X_train, X_test, y_train, y_test

# ============================================================
# 4.3 PATH B — Machine Learning Feature Engineering
# ============================================================

def compute_kinematics(file_data):
    """
    Compute velocity, acceleration, jerk from x, y, timestamp.
    """
    df = file_data.sort_values('timestamp').copy()

    dt = df['timestamp'].diff().fillna(1e-6)
    dt = dt.replace(0, 1e-6)

    dx = df['x'].diff().fillna(0)
    dy = df['y'].diff().fillna(0)

    df['vx'] = dx / dt
    df['vy'] = dy / dt
    df['velocity'] = np.sqrt(df['vx']**2 + df['vy']**2)

    dvx = df['vx'].diff().fillna(0)
    dvy = df['vy'].diff().fillna(0)
    df['ax'] = dvx / dt
    df['ay'] = dvy / dt
    df['acceleration'] = np.sqrt(df['ax']**2 + df['ay']**2)

    dax = df['ax'].diff().fillna(0)
    day = df['ay'].diff().fillna(0)
    df['jx'] = dax / dt
    df['jy'] = day / dt
    df['jerk'] = np.sqrt(df['jx']**2 + df['jy']**2)

    return df

def engineer_features(file_data):
    """
    Transform a single SVC file's time-series into a fixed-length feature vector.
    """
    df = compute_kinematics(file_data)

    features = {}

    for col in ['velocity', 'acceleration', 'jerk', 'pressure']:
        vals = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(vals) == 0:
            vals = pd.Series([0.0])

        features[f'{col}_mean'] = vals.mean()
        features[f'{col}_median'] = vals.median()
        features[f'{col}_std'] = vals.std()
        features[f'{col}_max'] = vals.max()
        features[f'{col}_min'] = vals.min()
        features[f'{col}_range'] = vals.max() - vals.min()
        features[f'{col}_25pct'] = vals.quantile(0.25)
        features[f'{col}_75pct'] = vals.quantile(0.75)
        features[f'{col}_iqr'] = vals.quantile(0.75) - vals.quantile(0.25)

    total_time = df['timestamp'].max() - df['timestamp'].min()
    features['total_duration'] = total_time if total_time > 0 else 1e-6

    in_air_mask = df['button_state'] == 0
    on_surface_mask = df['button_state'] == 1

    in_air_time = df.loc[in_air_mask, 'timestamp'].diff().fillna(0).sum()
    on_surface_time = df.loc[on_surface_mask, 'timestamp'].diff().fillna(0).sum()

    features['in_air_time'] = in_air_time
    features['on_surface_time'] = on_surface_time
    features['in_air_ratio'] = in_air_time / features['total_duration']
    features['on_surface_ratio'] = on_surface_time / features['total_duration']

    features['n_in_air_points'] = in_air_mask.sum()
    features['n_on_surface_points'] = on_surface_mask.sum()

    transitions = df['button_state'].diff().fillna(0)
    features['stroke_count'] = (transitions == 1).sum()

    on_surface_pressure = df.loc[on_surface_mask, 'pressure']
    if len(on_surface_pressure) > 0:
        features['on_surface_pressure_mean'] = on_surface_pressure.mean()
        features['on_surface_pressure_std'] = on_surface_pressure.std()
    else:
        features['on_surface_pressure_mean'] = 0.0
        features['on_surface_pressure_std'] = 0.0

    dx = df['x'].diff().fillna(0)
    dy = df['y'].diff().fillna(0)
    features['total_path_length'] = np.sqrt(dx**2 + dy**2).sum()

    features['n_points'] = len(df)

    return features

def prepare_ml_data(full_df, file_df, train_idx, test_idx, label_col):
    """
    Full Path B pipeline for a single fold.
    """
    train_files = file_df.iloc[train_idx]['file_name'].values
    test_files = file_df.iloc[test_idx]['file_name'].values

    train_features = []
    for fname in train_files:
        fdata = full_df[full_df['file_name'] == fname]
        feat = engineer_features(fdata)
        feat['file_name'] = fname
        train_features.append(feat)

    test_features = []
    for fname in test_files:
        fdata = full_df[full_df['file_name'] == fname]
        feat = engineer_features(fdata)
        feat['file_name'] = fname
        test_features.append(feat)

    train_feat_df = pd.DataFrame(train_features)
    test_feat_df = pd.DataFrame(test_features)

    feature_cols = [c for c in train_feat_df.columns if c != 'file_name']

    train_feat_df[feature_cols] = train_feat_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    test_feat_df[feature_cols] = test_feat_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_feat_df[feature_cols])
    X_test = scaler.transform(test_feat_df[feature_cols])

    y_train = file_df.iloc[train_idx][label_col].values.astype(np.int64)
    y_test = file_df.iloc[test_idx][label_col].values.astype(np.int64)

    print(f"  [ML] X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"  [ML] y_train dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  [ML] y_test  dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

    return X_train, X_test, y_train, y_test, feature_cols, scaler

# ============================================================
# 4.4  Modeling Methodology
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from itertools import product
import json
import time

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Install xgboost if needed ---
try:
    import xgboost as xgb
except ImportError:
    !pip install xgboost
    import xgboost as xgb

# ============================================================
# HYPERPARAMETER SEARCH SPACES
# ============================================================

# Path A: DL Hyperparameter Grid
DL_PARAM_GRID = {
    'hidden_size':  [64, 128, 256],
    'num_layers':   [1, 2, 3],
    'dropout':      [0.2, 0.3, 0.5],
    'lr':           [1e-3, 5e-4, 1e-4],
    'batch_size':   [16, 32, 64],
}

# Path B: RF Hyperparameter Grid
RF_PARAM_GRID = {
    'n_estimators':      [100, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
}

# Path B: XGBoost Hyperparameter Grid
XGB_PARAM_GRID = {
    'n_estimators':  [100, 300, 500],
    'max_depth':     [3, 6, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample':     [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

# Maximum number of random combos to try (budget cap)
MAX_DL_TRIALS  = 15
MAX_RF_TRIALS  = 20
MAX_XGB_TRIALS = 20


# ============================================================
# PATH A: Deep Learning Models (GRU & LSTM)
# ============================================================

class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=2, dropout=0.3):
        super(GRUClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=2, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, (_, _) = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out


def train_dl_model(model, X_train, y_train, X_test, y_test,
                   epochs=100, batch_size=32, lr=1e-3, patience=15):
    """
    Train a PyTorch DL model with class weighting and early stopping.
    Returns the trained model and training history.
    """
    model = model.to(device)

    classes = np.unique(y_train)
    cw = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = torch.FloatTensor(cw).to(device)
    print(f"  Class weights: {dict(zip(classes, cw.round(4)))}")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    train_dataset = TensorDataset(
        torch.FloatTensor(X_train),
        torch.LongTensor(y_train)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_tensor = torch.LongTensor(y_test).to(device)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_test_tensor)
            val_loss = criterion(val_outputs, y_test_tensor).item()

        scheduler.step(val_loss)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 20 == 0:
            print(f"    Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}")

        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history


def predict_dl_model(model, X_test):
    """Get predictions and probabilities from a DL model."""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_test).to(device)
        outputs = model(X_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
    return preds, probs


# ============================================================
# HYPERPARAMETER TUNING — DL (Random Search with inner val)
# ============================================================

def tune_dl_hyperparameters(model_class, model_name, X_train, y_train,
                            input_size, num_classes, max_trials=MAX_DL_TRIALS):
    """
    Random search over DL hyperparameters using a hold-out validation
    split carved from the training data (80/20 inner split).
    
    Returns: best_params dict
    """
    print(f"\n  {'~'*50}")
    print(f"  Tuning {model_name} — up to {max_trials} random trials")
    print(f"  {'~'*50}")

    # Inner train/val split (80/20 stratified)
    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    inner_train_idx, inner_val_idx = next(sss.split(X_train, y_train))

    X_inner_train = X_train[inner_train_idx]
    y_inner_train = y_train[inner_train_idx]
    X_inner_val   = X_train[inner_val_idx]
    y_inner_val   = y_train[inner_val_idx]

    # Generate random combinations
    rng = np.random.RandomState(RANDOM_STATE)
    all_combos = list(product(
        DL_PARAM_GRID['hidden_size'],
        DL_PARAM_GRID['num_layers'],
        DL_PARAM_GRID['dropout'],
        DL_PARAM_GRID['lr'],
        DL_PARAM_GRID['batch_size'],
    ))
    rng.shuffle(all_combos)
    combos_to_try = all_combos[:max_trials]

    best_f1 = -1
    best_params = {}
    tuning_log = []

    for trial_idx, (hs, nl, do, lr, bs) in enumerate(combos_to_try):
        params = {
            'hidden_size': hs, 'num_layers': nl,
            'dropout': do, 'lr': lr, 'batch_size': bs
        }
        print(f"\n    Trial {trial_idx+1}/{len(combos_to_try)}: {params}")

        try:
            model = model_class(
                input_size=input_size, hidden_size=hs,
                num_layers=nl, num_classes=num_classes, dropout=do
            )
            # Train with reduced epochs & patience for speed
            model, _ = train_dl_model(
                model, X_inner_train, y_inner_train,
                X_inner_val, y_inner_val,
                epochs=50, batch_size=bs, lr=lr, patience=8
            )
            preds, probs = predict_dl_model(model, X_inner_val)
            f1 = f1_score(y_inner_val, preds, zero_division=0)
            acc = np.mean(preds == y_inner_val)

            tuning_log.append({**params, 'val_f1': f1, 'val_acc': acc})
            print(f"    → Val F1 = {f1:.4f}, Val Acc = {acc:.4f}")

            if f1 > best_f1:
                best_f1 = f1
                best_params = params.copy()
                print(f"    ★ New best!")

        except Exception as e:
            print(f"    [SKIP] Error: {e}")
            tuning_log.append({**params, 'val_f1': 0.0, 'val_acc': 0.0, 'error': str(e)})

    print(f"\n  ✓ Best {model_name} params (Val F1 = {best_f1:.4f}): {best_params}")

    # Fallback to defaults if tuning failed entirely
    if not best_params:
        best_params = {
            'hidden_size': 128, 'num_layers': 2,
            'dropout': 0.3, 'lr': 1e-3, 'batch_size': 32
        }
        print(f"  ⚠ Tuning failed — using defaults: {best_params}")

    return best_params, tuning_log


# ============================================================
# HYPERPARAMETER TUNING — RF (Random Search with inner val)
# ============================================================

def tune_rf_hyperparameters(X_train, y_train, max_trials=MAX_RF_TRIALS):
    """
    Random search over RF hyperparameters using inner stratified split.
    Returns: best_params dict
    """
    print(f"\n  {'~'*50}")
    print(f"  Tuning Random Forest — up to {max_trials} random trials")
    print(f"  {'~'*50}")

    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    inner_train_idx, inner_val_idx = next(sss.split(X_train, y_train))

    X_inner_train = X_train[inner_train_idx]
    y_inner_train = y_train[inner_train_idx]
    X_inner_val   = X_train[inner_val_idx]
    y_inner_val   = y_train[inner_val_idx]

    rng = np.random.RandomState(RANDOM_STATE)
    all_combos = list(product(
        RF_PARAM_GRID['n_estimators'],
        RF_PARAM_GRID['max_depth'],
        RF_PARAM_GRID['min_samples_split'],
        RF_PARAM_GRID['min_samples_leaf'],
    ))
    rng.shuffle(all_combos)
    combos_to_try = all_combos[:max_trials]

    best_f1 = -1
    best_params = {}
    tuning_log = []

    for trial_idx, (n_est, md, mss, msl) in enumerate(combos_to_try):
        params = {
            'n_estimators': n_est, 'max_depth': md,
            'min_samples_split': mss, 'min_samples_leaf': msl
        }
        print(f"    Trial {trial_idx+1}/{len(combos_to_try)}: {params}", end="")

        try:
            rf = RandomForestClassifier(
                n_estimators=n_est, max_depth=md,
                min_samples_split=mss, min_samples_leaf=msl,
                class_weight='balanced',
                random_state=RANDOM_STATE, n_jobs=-1
            )
            rf.fit(X_inner_train, y_inner_train)
            preds = rf.predict(X_inner_val)
            f1 = f1_score(y_inner_val, preds, zero_division=0)
            acc = np.mean(preds == y_inner_val)

            tuning_log.append({**params, 'val_f1': f1, 'val_acc': acc})
            print(f"  → F1={f1:.4f}, Acc={acc:.4f}", end="")

            if f1 > best_f1:
                best_f1 = f1
                best_params = params.copy()
                print("  ★")
            else:
                print()

        except Exception as e:
            print(f"  [SKIP] {e}")
            tuning_log.append({**params, 'val_f1': 0.0, 'val_acc': 0.0, 'error': str(e)})

    print(f"\n  ✓ Best RF params (Val F1 = {best_f1:.4f}): {best_params}")

    if not best_params:
        best_params = {
            'n_estimators': 300, 'max_depth': None,
            'min_samples_split': 5, 'min_samples_leaf': 2
        }
        print(f"  ⚠ Tuning failed — using defaults: {best_params}")

    return best_params, tuning_log


# ============================================================
# HYPERPARAMETER TUNING — XGBoost (Random Search with inner val)
# ============================================================

def tune_xgb_hyperparameters(X_train, y_train, max_trials=MAX_XGB_TRIALS):
    """
    Random search over XGBoost hyperparameters using inner stratified split.
    Returns: best_params dict
    """
    print(f"\n  {'~'*50}")
    print(f"  Tuning XGBoost — up to {max_trials} random trials")
    print(f"  {'~'*50}")

    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    inner_train_idx, inner_val_idx = next(sss.split(X_train, y_train))

    X_inner_train = X_train[inner_train_idx]
    y_inner_train = y_train[inner_train_idx]
    X_inner_val   = X_train[inner_val_idx]
    y_inner_val   = y_train[inner_val_idx]

    # scale_pos_weight for the inner split
    classes, counts = np.unique(y_inner_train, return_counts=True)
    if len(classes) == 2:
        spw = counts[0] / counts[1]
    else:
        spw = 1.0

    rng = np.random.RandomState(RANDOM_STATE)
    all_combos = list(product(
        XGB_PARAM_GRID['n_estimators'],
        XGB_PARAM_GRID['max_depth'],
        XGB_PARAM_GRID['learning_rate'],
        XGB_PARAM_GRID['subsample'],
        XGB_PARAM_GRID['colsample_bytree'],
    ))
    rng.shuffle(all_combos)
    combos_to_try = all_combos[:max_trials]

    best_f1 = -1
    best_params = {}
    tuning_log = []

    for trial_idx, (n_est, md, lr, ss, csb) in enumerate(combos_to_try):
        params = {
            'n_estimators': n_est, 'max_depth': md,
            'learning_rate': lr, 'subsample': ss,
            'colsample_bytree': csb
        }
        print(f"    Trial {trial_idx+1}/{len(combos_to_try)}: {params}", end="")

        try:
            xgb_model = xgb.XGBClassifier(
                n_estimators=n_est, max_depth=md,
                learning_rate=lr, subsample=ss,
                colsample_bytree=csb,
                scale_pos_weight=spw,
                eval_metric='logloss',
                use_label_encoder=False,
                random_state=RANDOM_STATE, n_jobs=-1
            )
            xgb_model.fit(X_inner_train, y_inner_train, verbose=False)
            preds = xgb_model.predict(X_inner_val)
            f1 = f1_score(y_inner_val, preds, zero_division=0)
            acc = np.mean(preds == y_inner_val)

            tuning_log.append({**params, 'val_f1': f1, 'val_acc': acc})
            print(f"  → F1={f1:.4f}, Acc={acc:.4f}", end="")

            if f1 > best_f1:
                best_f1 = f1
                best_params = params.copy()
                print("  ★")
            else:
                print()

        except Exception as e:
            print(f"  [SKIP] {e}")
            tuning_log.append({**params, 'val_f1': 0.0, 'val_acc': 0.0, 'error': str(e)})

    print(f"\n  ✓ Best XGBoost params (Val F1 = {best_f1:.4f}): {best_params}")

    if not best_params:
        best_params = {
            'n_estimators': 300, 'max_depth': 6,
            'learning_rate': 0.1, 'subsample': 1.0,
            'colsample_bytree': 1.0
        }
        print(f"  ⚠ Tuning failed — using defaults: {best_params}")

    return best_params, tuning_log


# ============================================================
# MODEL BUILDERS USING TUNED PARAMS
# ============================================================

def build_tuned_dl_model(model_class, input_size, num_classes, best_params):
    """Instantiate a DL model with the best hyperparameters."""
    return model_class(
        input_size=input_size,
        hidden_size=best_params['hidden_size'],
        num_layers=best_params['num_layers'],
        num_classes=num_classes,
        dropout=best_params['dropout']
    )


def train_tuned_rf(X_train, y_train, best_params):
    """Train RF with tuned hyperparameters."""
    rf = RandomForestClassifier(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    return rf


def train_tuned_xgb(X_train, y_train, best_params):
    """Train XGBoost with tuned hyperparameters."""
    classes, counts = np.unique(y_train, return_counts=True)
    if len(classes) == 2:
        spw = counts[0] / counts[1]
    else:
        spw = 1.0

    xgb_model = xgb.XGBClassifier(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'],
        subsample=best_params['subsample'],
        colsample_bytree=best_params['colsample_bytree'],
        scale_pos_weight=spw,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train, verbose=False)
    return xgb_model


def predict_ml_model(model, X_test):
    """Get predictions and probabilities from an ML model."""
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)
    return preds, probs


# ============================================================
# 4.5  Performance Metrics
# ============================================================
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    accuracy_score
)
import matplotlib.pyplot as plt

def compute_fold_metrics(y_true, y_pred, y_prob, num_classes=2):
    """
    Compute all metrics for a single fold.
    Always includes accuracy.
    """
    metrics = {}

    cm = confusion_matrix(y_true, y_pred)
    metrics['confusion_matrix'] = cm

    # ---- Always compute accuracy ----
    metrics['accuracy'] = accuracy_score(y_true, y_pred)

    if num_classes == 2:
        tn, fp, fn, tp = cm.ravel()
        metrics['TP'] = tp
        metrics['TN'] = tn
        metrics['FP'] = fp
        metrics['FN'] = fn

        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1_score'] = f1_score(y_true, y_pred, zero_division=0)
        metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        if y_prob is not None and len(np.unique(y_true)) > 1:
            metrics['auc_roc'] = roc_auc_score(y_true, y_prob[:, 1])
        else:
            metrics['auc_roc'] = float('nan')
    else:
        metrics['precision'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['f1_score'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        if y_prob is not None and len(np.unique(y_true)) > 1:
            metrics['auc_roc'] = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
        else:
            metrics['auc_roc'] = float('nan')

    return metrics


def print_fold_metrics(metrics, fold_idx, model_name):
    """Pretty-print metrics for a single fold."""
    print(f"\n  --- {model_name} | Fold {fold_idx + 1} ---")
    print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1-Score  : {metrics['f1_score']:.4f}")
    print(f"  AUC-ROC   : {metrics['auc_roc']:.4f}")
    if 'specificity' in metrics:
        print(f"  Specificity: {metrics['specificity']:.4f}")
    print(f"  Confusion Matrix:\n{metrics['confusion_matrix']}")


def summarize_cv_results(all_fold_metrics, model_name, task_name):
    """
    Summarize metrics across K folds: report mean ± std.
    Always includes accuracy.
    """
    metric_keys = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']
    if 'specificity' in all_fold_metrics[0]:
        metric_keys.append('specificity')

    summary = {}
    print(f"\n{'='*60}")
    print(f"  {model_name} | {task_name} | 5-Fold CV Summary")
    print(f"{'='*60}")
    for key in metric_keys:
        values = [m[key] for m in all_fold_metrics if not np.isnan(m.get(key, float('nan')))]
        mean_val = np.mean(values) if values else float('nan')
        std_val = np.std(values) if values else float('nan')
        summary[f'{key}_mean'] = mean_val
        summary[f'{key}_std'] = std_val
        print(f"  {key:12s}: {mean_val:.4f} ± {std_val:.4f}")

    return summary


def plot_confusion_matrices(all_fold_metrics, model_name, task_name, class_names):
    """Plot confusion matrices for all folds."""
    fig, axes = plt.subplots(1, N_SPLITS, figsize=(4 * N_SPLITS, 4))
    for i, metrics in enumerate(all_fold_metrics):
        disp = ConfusionMatrixDisplay(
            confusion_matrix=metrics['confusion_matrix'],
            display_labels=class_names
        )
        disp.plot(ax=axes[i], cmap='Blues', values_format='d')
        axes[i].set_title(f'Fold {i+1}')
    fig.suptitle(f'{model_name} — {task_name}', fontsize=14)
    plt.tight_layout()
    plt.show()

# ============================================================
# MAIN EXPERIMENT LOOP (WITH HYPERPARAMETER TUNING)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

# Storage for all results
all_results = {}

# ============================================================
# Helper: run one full task (Detection or Staging)
# ============================================================
def run_experiment(task_name, file_df, label_col, folds, class_names):
    """
    Run the complete experiment pipeline for a given task,
    including per-fold hyperparameter tuning for all 4 models.
    """
    task_results = {}

    model_fold_metrics = {
        'GRU': [], 'LSTM': [], 'RF': [], 'XGBoost': []
    }
    best_fold_info = {
        'GRU': {'best_f1': -1}, 'LSTM': {'best_f1': -1},
        'RF': {'best_f1': -1}, 'XGBoost': {'best_f1': -1}
    }
    # Store all tuning logs
    all_tuning_logs = {
        'GRU': [], 'LSTM': [], 'RF': [], 'XGBoost': []
    }
    # Store best params per fold
    all_best_params = {
        'GRU': [], 'LSTM': [], 'RF': [], 'XGBoost': []
    }

    input_size = len(DL_FEATURES)
    num_classes = len(class_names)

    for fold_idx, (train_idx, test_idx) in enumerate(folds):
        print(f"\n{'#'*60}")
        print(f"# {task_name} — FOLD {fold_idx + 1}/{N_SPLITS}")
        print(f"{'#'*60}")

        # ---- Path A: Deep Learning Preprocessing ----
        print("\n  >> Preparing DL data...")
        X_train_dl, X_test_dl, y_train_dl, y_test_dl = prepare_dl_data(
            full_df, file_df, train_idx, test_idx, label_col
        )

        # ---- Path B: ML Feature Engineering ----
        print("\n  >> Preparing ML data...")
        X_train_ml, X_test_ml, y_train_ml, y_test_ml, feat_cols, ml_scaler = prepare_ml_data(
            full_df, file_df, train_idx, test_idx, label_col
        )

        # ================================================================
        # HYPERPARAMETER TUNING (per fold, using inner validation split)
        # ================================================================

        # ======== GRU Tuning ========
        print("\n  >> Tuning GRU hyperparameters...")
        gru_best_params, gru_tune_log = tune_dl_hyperparameters(
            GRUClassifier, 'GRU', X_train_dl, y_train_dl,
            input_size, num_classes, max_trials=MAX_DL_TRIALS
        )
        all_tuning_logs['GRU'].append(gru_tune_log)
        all_best_params['GRU'].append(gru_best_params)

        # ======== LSTM Tuning ========
        print("\n  >> Tuning LSTM hyperparameters...")
        lstm_best_params, lstm_tune_log = tune_dl_hyperparameters(
            LSTMClassifier, 'LSTM', X_train_dl, y_train_dl,
            input_size, num_classes, max_trials=MAX_DL_TRIALS
        )
        all_tuning_logs['LSTM'].append(lstm_tune_log)
        all_best_params['LSTM'].append(lstm_best_params)

        # ======== RF Tuning ========
        print("\n  >> Tuning Random Forest hyperparameters...")
        rf_best_params, rf_tune_log = tune_rf_hyperparameters(
            X_train_ml, y_train_ml, max_trials=MAX_RF_TRIALS
        )
        all_tuning_logs['RF'].append(rf_tune_log)
        all_best_params['RF'].append(rf_best_params)

        # ======== XGBoost Tuning ========
        print("\n  >> Tuning XGBoost hyperparameters...")
        xgb_best_params, xgb_tune_log = tune_xgb_hyperparameters(
            X_train_ml, y_train_ml, max_trials=MAX_XGB_TRIALS
        )
        all_tuning_logs['XGBoost'].append(xgb_tune_log)
        all_best_params['XGBoost'].append(xgb_best_params)

        # ================================================================
        # TRAIN FINAL MODELS WITH BEST PARAMS ON FULL FOLD TRAINING DATA
        # ================================================================

        # ======== GRU ========
        print(f"\n  >> Training GRU with best params: {gru_best_params}")
        gru_model = build_tuned_dl_model(
            GRUClassifier, input_size, num_classes, gru_best_params
        )
        gru_model, gru_history = train_dl_model(
            gru_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl,
            epochs=100,
            batch_size=gru_best_params['batch_size'],
            lr=gru_best_params['lr'],
            patience=15
        )
        gru_preds, gru_probs = predict_dl_model(gru_model, X_test_dl)
        gru_metrics = compute_fold_metrics(y_test_dl, gru_preds, gru_probs, num_classes)
        gru_metrics['best_params'] = gru_best_params
        print_fold_metrics(gru_metrics, fold_idx, 'GRU')
        model_fold_metrics['GRU'].append(gru_metrics)

        if gru_metrics['f1_score'] > best_fold_info['GRU']['best_f1']:
            best_fold_info['GRU'] = {
                'best_f1': gru_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': gru_model,
                'X_test': X_test_dl, 'y_test': y_test_dl,
                'y_pred': gru_preds, 'y_prob': gru_probs,
                'history': gru_history,
                'best_params': gru_best_params
            }

        # ======== LSTM ========
        print(f"\n  >> Training LSTM with best params: {lstm_best_params}")
        lstm_model = build_tuned_dl_model(
            LSTMClassifier, input_size, num_classes, lstm_best_params
        )
        lstm_model, lstm_history = train_dl_model(
            lstm_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl,
            epochs=100,
            batch_size=lstm_best_params['batch_size'],
            lr=lstm_best_params['lr'],
            patience=15
        )
        lstm_preds, lstm_probs = predict_dl_model(lstm_model, X_test_dl)
        lstm_metrics = compute_fold_metrics(y_test_dl, lstm_preds, lstm_probs, num_classes)
        lstm_metrics['best_params'] = lstm_best_params
        print_fold_metrics(lstm_metrics, fold_idx, 'LSTM')
        model_fold_metrics['LSTM'].append(lstm_metrics)

        if lstm_metrics['f1_score'] > best_fold_info['LSTM']['best_f1']:
            best_fold_info['LSTM'] = {
                'best_f1': lstm_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': lstm_model,
                'X_test': X_test_dl, 'y_test': y_test_dl,
                'y_pred': lstm_preds, 'y_prob': lstm_probs,
                'history': lstm_history,
                'best_params': lstm_best_params
            }

        # ======== Random Forest ========
        print(f"\n  >> Training RF with best params: {rf_best_params}")
        rf_model = train_tuned_rf(X_train_ml, y_train_ml, rf_best_params)
        rf_preds, rf_probs = predict_ml_model(rf_model, X_test_ml)
        rf_metrics = compute_fold_metrics(y_test_ml, rf_preds, rf_probs, num_classes)
        rf_metrics['best_params'] = rf_best_params
        print_fold_metrics(rf_metrics, fold_idx, 'RF')
        model_fold_metrics['RF'].append(rf_metrics)

        if rf_metrics['f1_score'] > best_fold_info['RF']['best_f1']:
            best_fold_info['RF'] = {
                'best_f1': rf_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': rf_model,
                'X_train': X_train_ml, 'X_test': X_test_ml,
                'y_train': y_train_ml, 'y_test': y_test_ml,
                'y_pred': rf_preds, 'y_prob': rf_probs,
                'feature_names': feat_cols,
                'best_params': rf_best_params
            }

        # ======== XGBoost ========
        print(f"\n  >> Training XGBoost with best params: {xgb_best_params}")
        xgb_model = train_tuned_xgb(X_train_ml, y_train_ml, xgb_best_params)
        xgb_preds, xgb_probs = predict_ml_model(xgb_model, X_test_ml)
        xgb_metrics = compute_fold_metrics(y_test_ml, xgb_preds, xgb_probs, num_classes)
        xgb_metrics['best_params'] = xgb_best_params
        print_fold_metrics(xgb_metrics, fold_idx, 'XGBoost')
        model_fold_metrics['XGBoost'].append(xgb_metrics)

        if xgb_metrics['f1_score'] > best_fold_info['XGBoost']['best_f1']:
            best_fold_info['XGBoost'] = {
                'best_f1': xgb_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': xgb_model,
                'X_train': X_train_ml, 'X_test': X_test_ml,
                'y_train': y_train_ml, 'y_test': y_test_ml,
                'y_pred': xgb_preds, 'y_prob': xgb_probs,
                'feature_names': feat_cols,
                'best_params': xgb_best_params
            }

    # ---- Summarize all models ----
    print(f"\n\n{'*'*60}")
    print(f"* SUMMARY: {task_name}")
    print(f"{'*'*60}")

    summaries = {}
    for model_name in ['GRU', 'LSTM', 'RF', 'XGBoost']:
        summaries[model_name] = summarize_cv_results(
            model_fold_metrics[model_name], model_name, task_name
        )
        plot_confusion_matrices(
            model_fold_metrics[model_name], model_name, task_name, class_names
        )

    task_results['fold_metrics'] = model_fold_metrics
    task_results['summaries'] = summaries
    task_results['best_fold_info'] = best_fold_info
    task_results['tuning_logs'] = all_tuning_logs
    task_results['best_params'] = all_best_params

    return task_results


# ============================================================
# RUN DETECTION TASK
# ============================================================
print("\n" + "=" * 70)
print("   RUNNING DETECTION TASK: Healthy Control vs PD")
print("=" * 70)

detection_results = run_experiment(
    task_name='Detection',
    file_df=detect_file_df,
    label_col='detection_label',
    folds=detection_folds,
    class_names=['Healthy', 'PD']
)

all_results['Detection'] = detection_results

# ============================================================
# RUN STAGING TASK
# ============================================================
print("\n" + "=" * 70)
print("   RUNNING STAGING TASK: Early PD vs Moderate PD")
print("=" * 70)

staging_results = run_experiment(
    task_name='Staging',
    file_df=stage_file_df,
    label_col='staging_label',
    folds=staging_folds,
    class_names=['Early PD', 'Moderate PD']
)

all_results['Staging'] = staging_results

# ============================================================
# 4.5  Final Results Comparison Table (with Accuracy)
# ============================================================

def create_results_table(all_results):
    """Create a comprehensive results comparison table including accuracy."""
    rows = []
    for task_name, task_data in all_results.items():
        for model_name, summary in task_data['summaries'].items():
            row = {
                'Task': task_name,
                'Model': model_name,
                'Path': 'DL' if model_name in ['GRU', 'LSTM'] else 'ML'
            }
            for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']:
                mean = summary.get(f'{metric}_mean', float('nan'))
                std = summary.get(f'{metric}_std', float('nan'))
                row[f'{metric}'] = f"{mean:.4f} ± {std:.4f}"
            rows.append(row)

    results_df = pd.DataFrame(rows)
    print("\n" + "=" * 100)
    print("  COMPREHENSIVE RESULTS TABLE (with Accuracy)")
    print("=" * 100)
    print(results_df.to_string(index=False))
    return results_df

results_table = create_results_table(all_results)

# --- ROC Curves ---
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(task_results, task_name):
    """Plot ROC curves for the best fold of each model."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    colors = {'GRU': 'blue', 'LSTM': 'green', 'RF': 'red', 'XGBoost': 'orange'}

    for model_name, info in task_results['best_fold_info'].items():
        y_true = info['y_test']
        y_prob = info['y_prob']
        if y_prob is not None and len(np.unique(y_true)) > 1:
            fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=colors[model_name],
                    label=f'{model_name} (AUC = {roc_auc:.4f}, Fold {info["fold_idx"]+1})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {task_name} (Best Folds)')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_roc_curves(detection_results, 'Detection')
plot_roc_curves(staging_results, 'Staging')

# ============================================================
# 4.6  Model Interpretability (XAI)
# ============================================================

# Install required packages
!pip install shap lime captum -q

import shap
import lime
import lime.lime_tabular
from captum.attr import IntegratedGradients

# ============================================================
# PATH A: Integrated Gradients for GRU/LSTM
# ============================================================

def run_integrated_gradients(task_results, model_name, task_name, class_names):
    """
    Apply Integrated Gradients to the best-performing DL model fold.
    """
    info = task_results['best_fold_info'][model_name]
    model = info['model']
    X_test = info['X_test']
    y_test = info['y_test']
    y_pred = info['y_pred']

    print(f"\n{'='*60}")
    print(f"  Integrated Gradients — {model_name} | {task_name}")
    print(f"  Best Fold: {info['fold_idx'] + 1} (F1 = {info['best_f1']:.4f})")
    print(f"  Best Params: {info.get('best_params', 'N/A')}")
    print(f"{'='*60}")

    model.eval()
    model.to(device)

    ig = IntegratedGradients(model)

    n_samples = min(20, len(X_test))
    sample_indices = np.random.RandomState(42).choice(len(X_test), n_samples, replace=False)

    all_attributions = []

    for idx in sample_indices:
        input_tensor = torch.FloatTensor(X_test[idx:idx+1]).to(device)
        input_tensor.requires_grad = True

        target_class = int(y_pred[idx])
        baseline = torch.zeros_like(input_tensor).to(device)

        attribution = ig.attribute(
            input_tensor, baselines=baseline,
            target=target_class, n_steps=50
        )
        all_attributions.append(attribution.squeeze().cpu().detach().numpy())

    attributions = np.array(all_attributions)

    avg_attribution = np.mean(np.abs(attributions), axis=0)

    feature_names = DL_FEATURES

    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(avg_attribution.T, aspect='auto', cmap='hot', interpolation='nearest')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Feature')
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels(feature_names)
    ax.set_title(f'Temporal Saliency Heatmap — {model_name} | {task_name}\n'
                 f'(Avg |Attribution| over {n_samples} test samples, Best Fold {info["fold_idx"]+1})')
    plt.colorbar(im, ax=ax, label='Mean |Attribution|')
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(len(feature_names), 1, figsize=(14, 3 * len(feature_names)), sharex=True)
    for i, feat in enumerate(feature_names):
        axes[i].plot(avg_attribution[:, i], color='crimson', lw=1.5)
        axes[i].fill_between(range(avg_attribution.shape[0]), avg_attribution[:, i], alpha=0.3, color='crimson')
        axes[i].set_ylabel(f'|Attr| — {feat}')
        axes[i].grid(alpha=0.3)
    axes[-1].set_xlabel('Time Step')
    fig.suptitle(f'Per-Feature Temporal Attribution — {model_name} | {task_name}', fontsize=14)
    plt.tight_layout()
    plt.show()

    correct_mask = y_pred[sample_indices] == y_test[sample_indices]
    correct_indices = np.where(correct_mask)[0]

    if len(correct_indices) >= 2:
        n_show = min(4, len(correct_indices))
        fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show))
        if n_show == 1:
            axes = [axes]
        for i in range(n_show):
            si = correct_indices[i]
            true_label = class_names[y_test[sample_indices[si]]]
                        im = axes[i].imshow(attributions[si].T, aspect='auto', cmap='hot', interpolation='nearest')
            axes[i].set_ylabel('Feature')
            axes[i].set_yticks(range(len(feature_names)))
            axes[i].set_yticklabels(feature_names)
            axes[i].set_title(f'Sample {sample_indices[si]} — True: {true_label}')
            plt.colorbar(im, ax=axes[i])
        axes[-1].set_xlabel('Time Step')
        fig.suptitle(f'Individual Saliency Maps — {model_name} | {task_name}', fontsize=14)
        plt.tight_layout()
        plt.show()

    return attributions


# ============================================================
# PATH B: SHAP & LIME for RF/XGBoost
# ============================================================

def run_shap_analysis(task_results, model_name, task_name, class_names):
    """
    Apply SHAP to the best-performing ML model fold.
    """
    info = task_results['best_fold_info'][model_name]
    model = info['model']
    X_test = info['X_test']
    X_train = info['X_train']
    y_test = info['y_test']
    y_pred = info['y_pred']
    feature_names = info['feature_names']

    print(f"\n{'='*60}")
    print(f"  SHAP Analysis — {model_name} | {task_name}")
    print(f"  Best Fold: {info['fold_idx'] + 1} (F1 = {info['best_f1']:.4f})")
    print(f"  Best Params: {info.get('best_params', 'N/A')}")
    print(f"{'='*60}")

    if model_name in ['RF', 'XGBoost']:
        explainer = shap.TreeExplainer(model)
    else:
        background = shap.sample(pd.DataFrame(X_train, columns=feature_names), 100)
        explainer = shap.KernelExplainer(model.predict_proba, background)

    shap_values = explainer.shap_values(X_test)

    # --- Global Feature Importance (Bar Plot) ---
    print("\n  >> SHAP Global Feature Importance (Bar Plot)")
    fig, ax = plt.subplots(figsize=(10, 8))
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[1], X_test, feature_names=feature_names,
                          plot_type="bar", show=False, max_display=20)
    else:
        shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                          plot_type="bar", show=False, max_display=20)
    plt.title(f'SHAP Global Feature Importance — {model_name} | {task_name}')
    plt.tight_layout()
    plt.show()

    # --- SHAP Summary Plot (Beeswarm) ---
    print("\n  >> SHAP Summary Plot (Beeswarm)")
    fig, ax = plt.subplots(figsize=(10, 8))
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[1], X_test, feature_names=feature_names,
                          show=False, max_display=20)
    else:
        shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                          show=False, max_display=20)
    plt.title(f'SHAP Summary — {model_name} | {task_name}')
    plt.tight_layout()
    plt.show()

    # --- Local Explanation (Force Plot for a single sample) ---
    print("\n  >> SHAP Local Explanation (first test sample)")
    if isinstance(shap_values, list):
        shap.force_plot(
            explainer.expected_value[1],
            shap_values[1][0],
            pd.DataFrame(X_test, columns=feature_names).iloc[0],
            matplotlib=True, show=True
        )
    else:
        shap.force_plot(
            explainer.expected_value,
            shap_values[0],
            pd.DataFrame(X_test, columns=feature_names).iloc[0],
            matplotlib=True, show=True
        )

    return shap_values


def run_lime_analysis(task_results, model_name, task_name, class_names):
    """
    Apply LIME to the best-performing ML model fold.
    """
    info = task_results['best_fold_info'][model_name]
    model = info['model']
    X_train = info['X_train']
    X_test = info['X_test']
    y_test = info['y_test']
    y_pred = info['y_pred']
    feature_names = info['feature_names']

    print(f"\n{'='*60}")
    print(f"  LIME Analysis — {model_name} | {task_name}")
    print(f"  Best Fold: {info['fold_idx'] + 1} (F1 = {info['best_f1']:.4f})")
    print(f"  Best Params: {info.get('best_params', 'N/A')}")
    print(f"{'='*60}")

    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train,
        feature_names=feature_names,
        class_names=class_names,
        mode='classification',
        random_state=RANDOM_STATE
    )

    n_explain = min(3, len(X_test))
    for i in range(n_explain):
        true_label = class_names[y_test[i]]
        pred_label = class_names[y_pred[i]]

        print(f"\n  >> LIME Explanation for Test Sample {i}")
        print(f"     True: {true_label}, Predicted: {pred_label}")

        explanation = lime_explainer.explain_instance(
            X_test[i],
            model.predict_proba,
            num_features=15,
            top_labels=1
        )

        fig = explanation.as_pyplot_figure()
        fig.set_size_inches(10, 6)
        plt.title(f'LIME — {model_name} | {task_name}\nSample {i}: True={true_label}, Pred={pred_label}')
        plt.tight_layout()
        plt.show()

    return lime_explainer


# ============================================================
# RUN XAI FOR ALL MODELS — DETECTION TASK
# ============================================================
print("\n" + "=" * 70)
print("   XAI — DETECTION TASK")
print("=" * 70)

# DL Models
for dl_model_name in ['GRU', 'LSTM']:
    try:
        run_integrated_gradients(detection_results, dl_model_name, 'Detection', ['Healthy', 'PD'])
    except Exception as e:
        print(f"  [ERROR] IG for {dl_model_name}: {e}")

# ML Models
for ml_model_name in ['RF', 'XGBoost']:
    try:
        run_shap_analysis(detection_results, ml_model_name, 'Detection', ['Healthy', 'PD'])
    except Exception as e:
        print(f"  [ERROR] SHAP for {ml_model_name}: {e}")

    try:
        run_lime_analysis(detection_results, ml_model_name, 'Detection', ['Healthy', 'PD'])
    except Exception as e:
        print(f"  [ERROR] LIME for {ml_model_name}: {e}")

# ============================================================
# RUN XAI FOR ALL MODELS — STAGING TASK
# ============================================================
print("\n" + "=" * 70)
print("   XAI — STAGING TASK")
print("=" * 70)

# DL Models
for dl_model_name in ['GRU', 'LSTM']:
    try:
        run_integrated_gradients(staging_results, dl_model_name, 'Staging', ['Early PD', 'Moderate PD'])
    except Exception as e:
        print(f"  [ERROR] IG for {dl_model_name}: {e}")

# ML Models
for ml_model_name in ['RF', 'XGBoost']:
    try:
        run_shap_analysis(staging_results, ml_model_name, 'Staging', ['Early PD', 'Moderate PD'])
    except Exception as e:
        print(f"  [ERROR] SHAP for {ml_model_name}: {e}")

    try:
        run_lime_analysis(staging_results, ml_model_name, 'Staging', ['Early PD', 'Moderate PD'])
    except Exception as e:
        print(f"  [ERROR] LIME for {ml_model_name}: {e}")

# ============================================================
# FINAL: Save results to Drive (with Accuracy + Tuning Logs)
# ============================================================
SAVE_DIR = 'results/hyperparameter_tuning/run13/'
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) Save the main results table (includes accuracy)
results_table.to_csv(os.path.join(SAVE_DIR, 'cv_results_summary.csv'), index=False)
print(f"\nResults saved to {SAVE_DIR}cv_results_summary.csv")

# 2) Save detailed per-fold metrics (always includes accuracy)
for task_name, task_data in all_results.items():
    for model_name, fold_metrics_list in task_data['fold_metrics'].items():
        rows = []
        for fold_idx, metrics in enumerate(fold_metrics_list):
            row = {
                'task': task_name,
                'model': model_name,
                'fold': fold_idx + 1,
                'accuracy': metrics['accuracy'],
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1_score': metrics['f1_score'],
                'auc_roc': metrics['auc_roc'],
            }
            # Include specificity if available (binary tasks)
            if 'specificity' in metrics:
                row['specificity'] = metrics['specificity']
            # Include the best hyperparameters used for this fold
            if 'best_params' in metrics:
                for param_key, param_val in metrics['best_params'].items():
                    row[f'param_{param_key}'] = param_val
            rows.append(row)
        per_fold_df = pd.DataFrame(rows)
        fname = f'{task_name}_{model_name}_per_fold.csv'
        per_fold_df.to_csv(os.path.join(SAVE_DIR, fname), index=False)

print("All per-fold results saved (with accuracy and best params per fold).")

# 3) Save hyperparameter tuning logs
for task_name, task_data in all_results.items():
    for model_name, tune_logs_per_fold in task_data['tuning_logs'].items():
        all_log_rows = []
        for fold_idx, fold_log in enumerate(tune_logs_per_fold):
            for trial in fold_log:
                trial_row = {'task': task_name, 'model': model_name, 'fold': fold_idx + 1}
                trial_row.update(trial)
                all_log_rows.append(trial_row)
        if all_log_rows:
            tune_df = pd.DataFrame(all_log_rows)
            fname = f'{task_name}_{model_name}_tuning_log.csv'
            tune_df.to_csv(os.path.join(SAVE_DIR, fname), index=False)

print("All hyperparameter tuning logs saved.")

# 4) Save best hyperparameters summary
best_params_rows = []
for task_name, task_data in all_results.items():
    for model_name, params_per_fold in task_data['best_params'].items():
        for fold_idx, params in enumerate(params_per_fold):
            row = {
                'task': task_name,
                'model': model_name,
                'fold': fold_idx + 1
            }
            row.update(params)
            best_params_rows.append(row)

best_params_df = pd.DataFrame(best_params_rows)
best_params_df.to_csv(os.path.join(SAVE_DIR, 'best_hyperparameters.csv'), index=False)
print(f"Best hyperparameters saved to {SAVE_DIR}best_hyperparameters.csv")

# 5) Print final summary of best params across all folds
print(f"\n{'='*80}")
print("  BEST HYPERPARAMETERS SUMMARY")
print(f"{'='*80}")
for task_name, task_data in all_results.items():
    print(f"\n  >> {task_name}")
    for model_name in ['GRU', 'LSTM', 'RF', 'XGBoost']:
        best_info = task_data['best_fold_info'][model_name]
        print(f"     {model_name:8s} | Best Fold: {best_info['fold_idx']+1} | "
              f"F1: {best_info['best_f1']:.4f} | "
              f"Params: {best_info.get('best_params', 'N/A')}")

print(f"\n\n✅ EXPERIMENT COMPLETE — All results, tuning logs, and accuracy metrics saved to {SAVE_DIR}")